# 街景字符识别 - FPN Multi-Head 模型

基于 PyTorch 的街景字符识别项目，采用 FPN + Multi-Head Attention 架构。

**支持三种模型**：`fpn_multihead`（默认）、`transformer`、`ctc`

**运行环境**：ModelScope PAI-DSW（8核CPU / 32GB内存 / 24GB显存）

**日志路径**：`{项目目录}/logs/train_YYYYMMDD_HHMMSS.log`

In [ ]:
# 环境配置：设置路径、随机种子、检测GPU、打印版本信息
import torch as t
import sys
import os

if os.path.dirname(os.path.abspath('__file__' if '__file__' in dir() else '.')) not in sys.path:
    sys.path.insert(0, os.path.dirname(os.path.abspath('__file__' if '__file__' in dir() else '.')))
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from utils.seed import set_seed
from config import config, BASE_DIR, data_dir, IS_MODELSCOPE

set_seed(42)

print('=' * 60)
print('环境信息')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'PyTorch: {t.__version__}')
print(f'CUDA available: {t.cuda.is_available()}')
if t.cuda.is_available():
    print(f'CUDA version: {t.version.cuda}')
    print(f'GPU: {t.cuda.get_device_name(0)}')
    vram = t.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'GPU VRAM: {vram:.1f} GB')
    t.set_float32_matmul_precision('high')
    print('TF32 matmul precision: enabled')
print(f'BASE_DIR: {BASE_DIR}')
print(f'ModelScope环境: {IS_MODELSCOPE}')
print(f'Model type: {config.model_type}')
print(f'Batch size: {config.batch_size}')
print(f'NUM_WORKERS: {os.cpu_count()} CPUs, workers={__import__("config").NUM_WORKERS}')
print('=' * 60)

In [ ]:
# 下载数据集（仅在数据不存在时执行）
from data.download import download_dataset
download_dataset()

## 数据探索

In [ ]:
# 统计训练集、验证集、测试集的图片数量
from glob import glob

train_list = glob(data_dir['train_data'] + '*.png')
val_list = glob(data_dir['val_data'] + '*.png')
test_list = glob(data_dir['test_data'] + '*.png')

print(f'训练集图片数: {len(train_list)}')
print(f'验证集图片数: {len(val_list)}')
print(f'测试集图片数: {len(test_list)}')

In [ ]:
# 分析训练集中数字位数的分布
import json

with open(data_dir['train_label'], 'r') as f:
    marks = json.load(f)

dicts = {}
for img, mark in marks.items():
    n = len(mark['label'])
    dicts[n] = dicts.get(n, 0) + 1
for k, v in sorted(dicts.items()):
    print(f'{k}位数字的图片数目: {v}')

## 模型定义

创建模型并统计参数量

In [ ]:
# 创建模型并统计参数量
from models import create_model

model = create_model('fpn_multihead')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型类型: fpn_multihead')
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print(f'模型大小: {total_params * 4 / (1024**2):.1f} MB (FP32)')

## 训练 FPN Multi-Head 模型

训练过程中日志会自动记录到 `logs/` 目录下，包含：
- 每个batch的loss、学习率、精度、GPU/CPU内存使用
- 每个epoch的训练/验证精度、耗时、early stopping状态

In [ ]:
# 训练FPN Multi-Head模型
# 日志自动保存到 {项目目录}/logs/ 目录
from trainer.multihead import MultiHeadTrainer
from utils.misc import find_latest_checkpoint

latest_checkpoint = find_latest_checkpoint(config.checkpoints)
if latest_checkpoint:
    print(f'发现检查点: {latest_checkpoint}')
    config.pretrained = latest_checkpoint

trainer = MultiHeadTrainer(model_type='fpn_multihead')
trainer.train()

## 评估

In [ ]:
# 在验证集上评估模型精度
val_acc = trainer._eval()
print(f'最佳验证精度: {trainer.best_acc * 100:.2f}%')

In [ ]:
# 逐Head评估每个数字位的分类精度
trainer.eval_detailed()

In [ ]:
# 使用多尺度测试时增强(TTA)评估
# TTA尺寸: [288, 320, 352, 384, 416]
tta_acc = trainer.eval_tta()
if tta_acc > trainer.best_acc:
    print(f'TTA提升精度: {tta_acc * 100:.2f}% > {trainer.best_acc * 100:.2f}%')

## 预测与提交

生成提交CSV文件，保存到 BASE_DIR 目录下

In [ ]:
# 使用最佳模型生成预测结果
import os
from inference.predict import predicts

predicts(trainer.best_checkpoint_path, os.path.join(BASE_DIR, 'result.csv'), use_tta=True, model_type='fpn_multihead')

## CTC 模型（可选）

CTC模型使用连接时序分类，与Multi-Head模型架构不同，可用于跨模型集成

In [ ]:
# 训练CTC模型并生成预测
from trainer.ctc import CTCTrainer
from inference.predict import ctc_predict

ctc_trainer = CTCTrainer()
ctc_trainer.train()

ctc_acc = ctc_trainer._eval()
print(f'CTC最佳验证精度: {ctc_trainer.best_acc * 100:.2f}%')

ctc_predict(ctc_trainer.best_checkpoint_path, os.path.join(BASE_DIR, 'result_ctc.csv'), use_tta=False)

## 跨模型集成（可选）

将Multi-Head模型和CTC模型的预测结果进行集成，通常能获得更好的精度

In [ ]:
# 跨模型集成：Multi-Head + CTC
from inference.predict import cross_model_ensemble

cross_model_ensemble(
    trainer.best_checkpoint_path,
    ctc_trainer.best_checkpoint_path,
    os.path.join(BASE_DIR, 'result_ensemble.csv')
)

## 训练日志查看

日志文件保存在 `{项目目录}/logs/` 目录下，可下载查看详细训练过程

In [ ]:
# 查看最新的训练日志
import os
from config import SCRIPT_DIR

log_dir = os.path.join(SCRIPT_DIR, 'logs')
if os.path.exists(log_dir):
    log_files = sorted([f for f in os.listdir(log_dir) if f.startswith('train_') and f.endswith('.log')])
    if log_files:
        latest_log = os.path.join(log_dir, log_files[-1])
        print(f'日志文件路径: {latest_log}')
        print(f'日志文件大小: {os.path.getsize(latest_log) / 1024:.1f} KB')
        print()
        print('=== 最近50行日志 ===')
        with open(latest_log, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            for line in lines[-50:]:
                print(line.rstrip())
    else:
        print('未找到训练日志文件')
else:
    print('日志目录不存在，请先运行训练')